# core

> Fill in a module description here

In [ ]:
#| default_exp core

## Legacy OCR to refactor
```
"""OCR evaluation reports"""

# AUTOGENERATED! DO NOT EDIT! File to edit: ../nbs/03_ocr.ipynb.

# %% auto 0
__all__ = ['mistral_api_key', 'src_dir', 'log_dir', 'get_doc_subtype', 'clean_pdf_name', 'setup_output_dirs', 'get_pdfs_and_dir',
           'save_page_images', 'process_batch_results', 'create_batch_ocr_job', 'submit_and_monitor_batch_job',
           'download_and_parse_results', 'process_single_evaluation_batch', 'process_all_reports_batch',
           'ocr_evaluation']

# %% ../nbs/03_ocr.ipynb 2
import os
import re
from io import BytesIO
from dotenv import load_dotenv
import base64
from fastcore.all import *
from rich import print
import urllib.parse
from pathlib import Path
from tqdm import tqdm
import json
import logging
import tempfile
from PIL import Image
import time

from mistralai import Mistral
from .readers import load_evals

# %% ../nbs/03_ocr.ipynb 3
load_dotenv()
mistral_api_key = os.getenv("MISTRAL_API_KEY")

# %% ../nbs/03_ocr.ipynb 4
src_dir = Path("../_data/")

# %% ../nbs/03_ocr.ipynb 11
def get_doc_subtype(
    id:str, # ID of the evaluation
    fname:str, # Name of the file
    evals # Evaluations data
    )->str: # Document Subtype
    "Get Document Subtype for a given file in the evaluation dataset"
    eval_data = L(evals).filter(lambda x: x['id']==id)
    if not eval_data: return None
    
    docs = L(eval_data[0]['docs'])
    matches = docs.filter(lambda x: Path(x['File URL']).name==fname)
    return matches[0]['Document Subtype'] if matches else None

# %% ../nbs/03_ocr.ipynb 14
# Note: filename will be cleaned upstream in a next version
def clean_pdf_name(pdf_name: str) -> str:
    """
    Clean PDF name to create folder-friendly string.
    Removes special characters and spaces, replaces with underscores.
    """
    # Remove URL encoding
    pdf_name = urllib.parse.unquote(pdf_name)
    
    # Replace spaces and special characters with underscores
    # Replace any character that is not a word character (\w), whitespace (\s), or hyphen (-) with underscore
    cleaned = re.sub(r'[^\w\s-]', '_', pdf_name)
    
    # Replace any sequence of hyphens or whitespace with a single underscore
    cleaned = re.sub(r'[-\s]+', '_', cleaned)
    
    # Replace multiple consecutive underscores with a single underscore
    cleaned = re.sub(r'_+', '_', cleaned)
    cleaned = cleaned.strip('_')  # Remove leading/trailing underscores
    
    return cleaned.lower()

# %% ../nbs/03_ocr.ipynb 17
log_dir = Path.home() / '.evaluatr'
log_dir.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(str(log_dir / 'batch_ocr.log')),
        logging.StreamHandler()
    ]
)

# %% ../nbs/03_ocr.ipynb 18
def setup_output_dirs(md_library_path="../_data/md_library"):
    "Set up the output directory structure for markdown files"
    md_output_dir = Path(md_library_path)
    mkdir(md_output_dir, parents=True, exist_ok=True, overwrite=False)
    return md_output_dir

# %% ../nbs/03_ocr.ipynb 19
def get_pdfs_and_dir(
    report_path:Path, # Path to the report directory
    md_output_dir:Path # Path to the output directory
    ) -> tuple[list[Path], str]:
    "Get PDFs from report directory and create output directory"
    pdfs = report_path.ls(file_exts='.pdf')
    eval_report_path = report_path.name
    mkdir(md_output_dir / eval_report_path, parents=True, exist_ok=True, overwrite=False)
    return pdfs, eval_report_path

# %% ../nbs/03_ocr.ipynb 22
def save_page_images(page, dest_folder: Path):
    "Save all images from a page to destination folder as PNG"
    images = page.images if hasattr(page, 'images') else page.get('images', [])
    
    for img in images:
        img_data = getattr(img, 'image_base64', img.get('image_base64'))
        img_id = getattr(img, 'id', img.get('id'))
        
        if img_data and img_id:
            img_bytes = base64.b64decode(img_data.split(',')[1])
            pil_img = Image.open(BytesIO(img_bytes))
            output_path = dest_folder / img_id
            pil_img.save(output_path)

# %% ../nbs/03_ocr.ipynb 23
def process_batch_results(results, md_output_dir):
    "Process batch OCR results and save to appropriate folders"
    for result in results:
        try:
            # Parse custom_id to get eval_id and pdf_name
            eval_id, pdf_name = result['custom_id'].split('_', 1)
            
            # Get OCR response
            ocr_response = result['response']['body']
            
            # Create folder structure
            pdf_clean_name = clean_pdf_name(pdf_name)
            pdf_dir = md_output_dir / eval_id / pdf_clean_name
            pdf_dir.mkdir(parents=True, exist_ok=True)
            
            # Save each page markdown
            for page in ocr_response['pages']:
                page_num = page['index'] + 1
                page_path = pdf_dir / f"page_{page_num}.md"
                page_path.write_text(page['markdown'])
            
            # Save images if they exist
            img_dir = pdf_dir / 'img'
            for page in ocr_response['pages']:
                if page.get('images'):
                    img_dir.mkdir(parents=True, exist_ok=True)
                    save_page_images(page, img_dir)
            
            logging.info(f"Saved {len(ocr_response['pages'])} pages for {pdf_clean_name}")
            
        except Exception as e:
            logging.error(f"Error processing result {result.get('custom_id', 'unknown')}: {e}")

# %% ../nbs/03_ocr.ipynb 24
def create_batch_ocr_job(
    pdf_paths: List[Path],
    eval_report_path: str,
    model: str = "mistral-ocr-latest",
    include_images: bool = True,
    api_key: str = mistral_api_key
):
    "Create batch entries for PDFs from one evaluation report"
    cli = Mistral(api_key=api_key)
    
    batch_entries = []
    for pdf_path in pdf_paths:
        uploaded_pdf = cli.files.upload(
            file={
                "file_name": pdf_path.stem,
                "content": pdf_path.read_bytes(),
            },
            purpose="ocr"
        )
        
        signed_url = cli.files.get_signed_url(file_id=uploaded_pdf.id)
        entry = {
            "custom_id": f"{eval_report_path}_{pdf_path.name}",
            "body": {
                "document": {
                    "type": "document_url",
                    "document_url": signed_url.url,
                },
                "include_image_base64": include_images
            }
        }
        batch_entries.append(entry)
        logging.info(f"Added {pdf_path.name} to batch for {eval_report_path}")
        
    return batch_entries, cli


# %% ../nbs/03_ocr.ipynb 25
def submit_and_monitor_batch_job(batch_entries, eval_report_path, cli):
    "Submit batch job and monitor until completion"
    with tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl', delete=True) as temp_file:
        # Write batch entries to temp file
        for entry in batch_entries:
            temp_file.write(json.dumps(entry) + '\n')
        temp_file.flush()
        
        # Upload and create job
        batch_data = cli.files.upload(
            file={"file_name": f"batch_{eval_report_path}.jsonl", 
                  "content": open(temp_file.name, "rb")},
            purpose="batch"
        )
        
        created_job = cli.batch.jobs.create(
            input_files=[batch_data.id],
            model="mistral-ocr-latest",
            endpoint="/v1/ocr"
        )
        
        logging.info(f"Batch job created for {eval_report_path}: {created_job.id}")
        
        # Monitor completion
        while True:
            job = cli.batch.jobs.get(job_id=created_job.id)
            logging.info(f"Job status: {job.status} - {job.succeeded_requests}/{job.total_requests} completed")
            
            if job.status not in ["QUEUED", "RUNNING"]:
                break
            time.sleep(10)
        
        return job

# %% ../nbs/03_ocr.ipynb 26
def download_and_parse_results(job, cli):
    "Download and parse batch job results"
    response = cli.files.download(file_id=job.output_file)
    content = response.read().decode('utf-8')
    
    results = []
    for line in content.strip().split('\n'):
        if line:
            results.append(json.loads(line))
    
    logging.info(f"Downloaded and parsed {len(results)} OCR results")
    return results

# %% ../nbs/03_ocr.ipynb 27
def process_single_evaluation_batch(report: Path, md_output_dir: Path):
    "Process one evaluation report using batch OCR"
    logging.info(f"Processing evaluation: {report.name}")
    pdfs, eval_report_path = get_pdfs_and_dir(report, md_output_dir)
    
    if not pdfs:
        logging.warning(f"No PDFs found in {eval_report_path}")
        return
    
    batch_entries, cli = create_batch_ocr_job(pdfs, eval_report_path)
    
    job = submit_and_monitor_batch_job(batch_entries, eval_report_path, cli)
    
    if job and job.status == "SUCCESS":
        results = download_and_parse_results(job, cli)
        process_batch_results(results, md_output_dir)
        logging.info(f"Completed processing evaluation: {eval_report_path}")
    else:
        logging.error(f"Job failed for {eval_report_path}")

# %% ../nbs/03_ocr.ipynb 28
def process_all_reports_batch(
    reports: list[Path],
    md_library_path="../_data/md_library"
):
    "Process evaluation reports using batch OCR"
    logging.info(f"Starting batch OCR processing for {len(reports)} reports")
    md_output_dir = setup_output_dirs(md_library_path)
    
    for report in tqdm(reports, desc="Processing reports"):
        process_single_evaluation_batch(report, md_output_dir)
    
    logging.info("Batch OCR processing completed for all reports")

# %% ../nbs/03_ocr.ipynb 32
@call_parse
def ocr_evaluation(
    eval_id: str,  # Evaluation ID to process
    pdf_dir: str = "../data/pdf_library",  # Directory containing PDF folders
    output_dir: str = "../data/md_library",  # Output directory for markdown
    overwrite: bool = False  # Overwrite if it exists
):  
    "Process OCR for a single evaluation report"
    eval_path = Path(pdf_dir) / eval_id
    
    if not eval_path.exists():
        logging.error(f"Evaluation directory not found: {eval_path}")
        return
    
    output_path = Path(output_dir) / eval_id
    if output_path.exists() and not overwrite:
        logging.info(f"Output already exists for {eval_id}. Use --overwrite to reprocess.")
        return
    
    # Get PDFs, excluding those with underscore prefix
    pdfs = [p for p in eval_path.glob("*.pdf") if not p.name.startswith("_")]
    
    if not pdfs:
        logging.warning(f"No PDFs found in {eval_path}")
        return
    
    logging.info(f"Found {len(pdfs)} PDF(s) to process in {eval_id}")
    
    md_output_dir = setup_output_dirs(output_dir)
    process_single_evaluation_batch(eval_path, md_output_dir)
```

My goal is to refactor this module the "Legacy OCR" but as a single package with no dependency on `evaluatr`. The idea is to pass a pdf file or a folder containing multiple ones and converting them using batch ocr. 

Can we go through this task together.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

I'd be happy to help you refactor this OCR module into a standalone package! 

Looking at the legacy code, the core functionality is:
1. Uploading PDFs to Mistral for batch OCR
2. Monitoring the batch job
3. Downloading and parsing results
4. Saving markdown and images to disk

The evaluatr-specific parts to remove are `get_doc_subtype` and `load_evals`.

Let's start fresh and work incrementally. What would you like to tackle first? I'd suggest we start with basic imports and a simple function to upload a single PDF, then build from there. What do you think?

In [ ]:
#| export
import os, re, json, time, base64, tempfile, logging
from io import BytesIO
from pathlib import Path
from dotenv import load_dotenv
from PIL import Image
from mistralai import Mistral

In [ ]:
#| export
#| export
def get_api_key(key=None):
    "Get Mistral API key from parameter or environment"
    key = key or os.getenv("MISTRAL_API_KEY")
    if not key: raise ValueError("MISTRAL_API_KEY not found")
    return key

In [ ]:
#| export
#| export
ocr_model = "mistral-ocr-latest"
ocr_endpoint = "/v1/ocr"

In [ ]:
#| export
#| export
def upload_pdf(path, key=None):
    "Upload PDF to Mistral and return signed URL"
    c = Mistral(api_key=get_api_key(key))
    path = Path(path)
    uploaded = c.files.upload(file=dict(file_name=path.stem, content=path.read_bytes()), purpose="ocr")
    return c.files.get_signed_url(file_id=uploaded.id).url, c

In [ ]:
#| eval: false
fname = 'files/test/attention-is-all-you-need.pdf'
url, cli = upload_pdf(fname)
url, cli


In [ ]:
#| export
#| export
def create_batch_entry(path, url, cid=None, inc_img=True):
    "Create a batch entry dict for OCR"
    path = Path(path)
    if not cid: cid = path.stem
    return dict(custom_id=cid, body=dict(document=dict(type="document_url", document_url=url), include_image_base64=inc_img))

In [ ]:
#| eval: false
entry = create_batch_entry(fname, url)
entry

In [ ]:
#| export
#| export
def prep_pdf_batch(
    path:str, # Path to PDF file    
    cid=None, # Custom ID for batch entry
    inc_img=True, # Include image in batch entry
    key=None # API key
    ) -> dict: # Batch entry dict
    "Upload PDF and create batch entry in one step"
    url, c = upload_pdf(path, key)
    return create_batch_entry(path, url, cid, inc_img), c

What's the purpose of "key"

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The `key` parameter allows you to optionally pass a Mistral API key directly to the function. If you don't provide it, the function will look for `MISTRAL_API_KEY` in your environment variables (via `get_api_key`). 

This gives flexibility - you can either set the key once in your environment, or pass different keys for different calls if needed.

In [ ]:
#| eval: false
entry, c = prep_pdf_batch(fname)
entry

In [ ]:
#| export
#| export
def submit_batch(entries, c, model=ocr_model, endpoint=ocr_endpoint):
    "Submit batch entries and return job"
    with tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl', delete=True) as f:
        for e in entries: f.write(json.dumps(e) + '\n')
        f.flush()
        batch_data = c.files.upload(file=dict(file_name="batch.jsonl", content=open(f.name, "rb")), purpose="batch")
    return c.batch.jobs.create(input_files=[batch_data.id], model=model, endpoint=endpoint)

In [ ]:
#| eval: false
job = submit_batch([entry], c)
job

Perfect, now we now:
- how to upload a pdf
- how to prepare a batch and submit it

What's next?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Next we need to monitor the batch job until it completes, then download and parse the results. Let's add a function to poll the job status:

```python
#| export
def wait_for_job(job, c, sleep=10, verbose=True):
    "Poll job until completion"
    while job.status in ["QUEUED", "RUNNING"]:
        if verbose: print(f"Status: {job.status} - {job.succeeded_requests}/{job.total_requests}")
        time.sleep(sleep)
        job = c.batch.jobs.get(job_id=job.id)
    return job
```

In [ ]:
#| export
def wait_for_job(job, c, poll_interval=10):
    "Poll job until completion and return final job status"
    while job.status in ["QUEUED", "RUNNING"]:
        time.sleep(poll_interval)
        job = c.batch.jobs.get(job_id=job.id)
    return job

In [ ]:
#| eval: false
final_job = wait_for_job(job, c)
final_job.status, final_job.succeeded_requests, final_job.total_requests

In [ ]:
#| export
def download_results(job, c):
    "Download and parse batch job results"
    content = c.files.download(file_id=job.output_file).read().decode('utf-8')
    return [json.loads(line) for line in content.strip().split('\n') if line]

In [ ]:
#| eval: false
results = download_results(final_job, c)
len(results), results[0].keys()

In [ ]:
#| export
def get_ocr_response(result):
    "Extract OCR response from batch result"
    return result['response']['body']

In [ ]:
#| eval: false
ocr_resp = get_ocr_response(results[0])
len(ocr_resp['pages']), ocr_resp['pages'][0].keys()

In [ ]:
#| export
def save_images(page, img_dir='img'):
    "Save all images from a page to directory"
    for img in page.get('images', []):
        if img.get('image_base64') and img.get('id'):
            img_bytes = base64.b64decode(img['image_base64'].split(',')[1])
            Image.open(BytesIO(img_bytes)).save(img_dir / img['id'])

In [ ]:
#| export
def save_page(page, out_dir, img_dir='img'):
    "Save single page markdown and images"
    (out_dir / f"page_{page['index']+1}.md").write_text(page['markdown'])
    if page.get('images'):
        img_dir.mkdir(exist_ok=True)
        save_images(page, img_dir)

In [ ]:
#| export
def save_pages(ocr_resp, out_dir, cid):
    "Save markdown pages and images from OCR response to output directory"
    out_dir = Path(out_dir) / cid
    out_dir.mkdir(parents=True, exist_ok=True)
    img_dir = out_dir / 'img'
    for page in ocr_resp['pages']: save_page(page, out_dir, img_dir)
    return out_dir

In [ ]:
#| eval: false
save_pages(ocr_resp, 'files/test/md', results[0]['custom_id'])

In [ ]:
#| eval: false
%ls files/test/md/attention-is-all-you-need

In [ ]:
#| eval: false
%ls -R files/test/md

In [ ]:
#| export
def ocr_pdf(path, out_dir='md', inc_img=True, key=None, poll_interval=2):
    "OCR a PDF file or folder of PDFs and save results"
    path = Path(path)
    if path.is_file(): pdfs = [path]
    elif path.is_dir(): pdfs = path.ls(file_exts='.pdf')
    else: raise ValueError(f"Path not found: {path}")
    if not pdfs: raise ValueError(f"No PDFs found in {path}")
    entries, c = [], None
    for pdf in pdfs:
        entry, c = prep_pdf_batch(pdf, inc_img=inc_img, key=key)
        entries.append(entry)
    job = submit_batch(entries, c)
    job = wait_for_job(job, c, poll_interval)
    if job.status != 'SUCCESS': raise Exception(f"Job failed with status: {job.status}")
    results = download_results(job, c)
    return [save_pages(get_ocr_response(r), out_dir, r['custom_id']) for r in results]

In [ ]:
#| eval: false
ocr_pdf('files/test', 'files/test/md_all')

In [ ]:
#| eval: false
%ls -R files/test/md_all